# Notebook 09: Decoder Phase Diagram (Matrix C)

Artifact-only decoder comparison by structure regime, with explicit placeholders for unavailable decoder slices.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    from notebooks._helpers import (
        repo_root,
        ensure_numeric_columns,
        load_matrix_report,
        load_matrix_summary,
        load_metrics_master,
        load_run_manifest,
        ensure_run_status_norm,
        parse_metric_availability,
        ensure_optional_columns,
        placeholder_plot,
        unavailable_or_numeric,
        unavailable_panel_table,
    )
except ModuleNotFoundError:
    from _helpers import (
        repo_root,
        ensure_numeric_columns,
        load_matrix_report,
        load_matrix_summary,
        load_metrics_master,
        load_run_manifest,
        ensure_run_status_norm,
        parse_metric_availability,
        ensure_optional_columns,
        placeholder_plot,
        unavailable_or_numeric,
        unavailable_panel_table,
    )

from src.notebook_utils import FilterPipeline

ROOT = repo_root()

pd.set_option("display.max_columns", 120)



In [ ]:
run_manifest = load_run_manifest(ROOT / "results", required=False)
metrics_master = load_metrics_master(ROOT / "results", required=False)

results_root = ROOT / "results"
master_parquet_path = results_root / "metrics_master.parquet"
master_json_path = results_root / "metrics_master.json"
master_artifacts_available = master_parquet_path.exists() or master_json_path.exists()

source_details = {
    "preferred": "metrics_master(decoder-study-compatible rows)",
    "fallback": "matrix_c_summary.csv + matrix_c_report.json",
    "active": "none",
    "execution_filter": "mixture_only",
    "master_artifacts_available": bool(master_artifacts_available),
    "stage_filter": "not_evaluated",
}

# Preferred source: canonical metrics_master.
matrix_c = pd.DataFrame()
if not metrics_master.empty:
    candidate = metrics_master.copy()
    pipe = FilterPipeline(candidate, label="decoder_study_input")

    stage_filter_reason = "stage_missing_using_matrix_family"
    if "stage" in candidate.columns:
        stage_norm = candidate["stage"].astype(str).str.strip().str.lower()
        stage_match = stage_norm.isin({"decoder_study", "matrix_c"})
        stage_missing = candidate["stage"].isna() | stage_norm.isin({"", "nan", "none"})
        if stage_match.any():
            # Keep decoder-study rows and tolerate missing stage values from mixed producer schemas.
            pipe = pipe.filter(
                (stage_match | stage_missing).reindex(pipe.result().index, fill_value=False),
                label="stage_filter", reason="stage in {decoder_study, matrix_c} with null tolerance"
            )
            stage_filter_reason = "stage_match_with_null_tolerance"
        else:
            # Stage column exists but has no decoder-study labels in this snapshot; rely on matrix/family guards.
            stage_filter_reason = "stage_present_no_decoder_values_using_matrix_family"
    source_details["stage_filter"] = stage_filter_reason

    if "matrix" in candidate.columns:
        pipe = pipe.filter(
            pipe.result()["matrix"].astype(str).eq("matrix_c"),
            label="matrix_filter", reason="matrix == matrix_c"
        )
    if "family" in candidate.columns:
        pipe = pipe.filter(
            pipe.result()["family"].astype(str).isin(["P", "S"]),
            label="family_filter", reason="family in {P, S}"
        )

    matrix_c = pipe.result()
    source_details["active"] = "metrics_master"
elif master_artifacts_available:
    # Master artifact exists but produced no rows; do not fall back to legacy tables.
    source_details["active"] = "metrics_master"

# Fallback only when master artifacts are unavailable.
if metrics_master.empty and not master_artifacts_available:
    summary_path = ROOT / "results" / "tables" / "matrix_c_summary.csv"
    report_path = ROOT / "results" / "reports" / "matrix_c_report.json"
    summary_df = load_matrix_summary("matrix_c", ROOT / "results") if summary_path.exists() else pd.DataFrame()
    report_payload = load_matrix_report("matrix_c", ROOT / "results") if report_path.exists() else {}
    report_rows = pd.DataFrame(report_payload.get("records") or report_payload.get("rows") or [])

    if not summary_df.empty and not report_rows.empty and "run_id" in summary_df.columns and "run_id" in report_rows.columns:
        enrich_cols = ["run_id"] + [c for c in report_rows.columns if c not in summary_df.columns]
        matrix_c = summary_df.merge(report_rows[enrich_cols], on="run_id", how="left")
    elif not summary_df.empty:
        matrix_c = summary_df.copy()
    else:
        matrix_c = report_rows.copy()

    if not matrix_c.empty:
        source_details["active"] = "legacy_matrix_c"

matrix_c = ensure_optional_columns(
    matrix_c,
    object_defaults={"metric_availability": "{}"},
)
matrix_c["metric_availability_dict"] = matrix_c["metric_availability"].apply(parse_metric_availability)
matrix_c = ensure_run_status_norm(matrix_c)

if "execution_mode" in matrix_c.columns:
    matrix_c["execution_mode_norm"] = (
        matrix_c["execution_mode"].astype(str).str.strip().str.lower().replace({"": "unavailable", "nan": "unavailable"})
    )
else:
    matrix_c["execution_mode_norm"] = "unavailable"

# Guard optional decoder-study numeric fields across mixed producer schemas.
optional_decoder_numeric_cols = [
    "structure_collision_rate",
    "structure_rank_deficiency",
    "structure_ambiguous_syndrome_count",
    "structure_short_cycle_proxy",
    "structure_distance_proxy",
    "gain_over_bp1",
    "gain_over_bruteforce",
    "best_sampled_F",
    "postselection_success",
]
for _col in optional_decoder_numeric_cols:
    if _col not in matrix_c.columns:
        matrix_c[_col] = np.nan
    matrix_c[_col] = pd.to_numeric(matrix_c[_col], errors="coerce")

if matrix_c.empty:
    focus = pd.DataFrame(columns=matrix_c.columns)
else:
    exec_pipe = FilterPipeline(matrix_c, label="execution_mode_input")
    exec_pipe = exec_pipe.filter(
        exec_pipe.result()["execution_mode_norm"] == "mixture",
        label="execution_mode", reason="execution_mode_norm == mixture"
    )
    mixture_focus = exec_pipe.result()
    if mixture_focus.empty and "execution_mode" not in matrix_c.columns:
        focus = matrix_c.copy()
        source_details["execution_filter"] = "execution_mode_missing_using_all_rows"
    else:
        focus = mixture_focus

for _col in optional_decoder_numeric_cols:
    if _col not in focus.columns:
        focus[_col] = np.nan
    focus[_col] = pd.to_numeric(focus[_col], errors="coerce")
for col, fill in [("run_status", np.nan), ("family", "unknown"), ("decoder", "unknown")]:
    if col not in focus.columns:
        focus[col] = fill

focus = ensure_run_status_norm(focus)
if "execution_mode_norm" not in focus.columns:
    focus["execution_mode_norm"] = (
        focus.get("execution_mode", pd.Series(index=focus.index, dtype=object))
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"": "unavailable", "nan": "unavailable"})
    )

display(pd.DataFrame([{
    "rows": len(focus),
    "statuses": dict(focus["run_status"].value_counts(dropna=False)) if not focus.empty else {},
    "statuses_norm": dict(focus["run_status_norm"].value_counts(dropna=False)) if (not focus.empty and "run_status_norm" in focus.columns) else {},
    "decoders": dict(focus["decoder"].astype(str).value_counts()) if not focus.empty else {},
    "families": dict(focus["family"].astype(str).value_counts()) if not focus.empty else {},
    "source_record_count": len(matrix_c),
    "master_rows": len(metrics_master),
    "source_active": source_details["active"],
    "execution_filter": source_details["execution_filter"],
    "master_artifacts_available": source_details["master_artifacts_available"],
    "stage_filter": source_details["stage_filter"],
}]))

if focus.empty:
    if source_details["active"] == "metrics_master" and master_artifacts_available:
        reason = "metrics_master artifact is available, but no decoder-study-compatible matrix_c rows matched this snapshot."
    elif source_details["active"] == "legacy_matrix_c":
        reason = "Legacy matrix_c fallback loaded, but no usable rows matched this snapshot."
    else:
        reason = "No matrix_c decoder-study rows found in canonical or legacy artifacts."
    display(unavailable_panel_table(
        panel="Decoder phase diagram input",
        reason=reason,
        details=f"active_source={source_details['active']}; execution_filter={source_details['execution_filter']}; stage_filter={source_details['stage_filter']}; master_available={source_details['master_artifacts_available']}",
    ))


## Decoder Availability by Family/Status


In [ ]:
if focus.empty:
    coverage = pd.DataFrame()
    display(unavailable_panel_table(
        panel="Coverage table",
        reason="No mixture rows available for matrix_c family P/S.",
    ))
else:
    coverage_focus = focus.copy()
    for col, fill in [("family", "unknown"), ("decoder", "unknown"), ("run_status_norm", "unavailable")]:
        if col not in coverage_focus.columns:
            coverage_focus[col] = fill
        coverage_focus[col] = coverage_focus[col].fillna(fill)

    coverage = (
        coverage_focus.groupby(["family", "decoder", "run_status_norm"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values(["family", "decoder", "run_status_norm"])
    )
    display(coverage)

    unavailable_decoders = sorted(
        coverage_focus.groupby("decoder")["run_status_norm"].apply(lambda s: "completed" not in set(s)).pipe(lambda x: x[x].index.tolist())
    )
    if unavailable_decoders:
        display(unavailable_panel_table(
            panel="Decoder availability",
            reason="Decoder has no completed rows in current artifacts.",
            details=", ".join(unavailable_decoders),
        ))


## Decoder Comparison by Structure Regime


In [ ]:
required_numeric_cols = [
    "structure_collision_rate",
    "structure_rank_deficiency",
    "best_sampled_F",
    "postselection_success",
    "gain_over_bp1",
    "gain_over_bruteforce",
    "structure_ambiguous_syndrome_count",
    "structure_short_cycle_proxy",
    "structure_distance_proxy",
]

if focus.empty:
    ok_rows = pd.DataFrame(columns=required_numeric_cols + ["run_id", "decoder", "structure_regime"])
    panel = pd.DataFrame()
    display(unavailable_panel_table(
        panel="Structure regime comparison",
        reason="No decoder-study mixture rows available for matrix_c family P/S.",
    ))
else:
    focus_safe = focus.copy()
    focus_safe = ensure_run_status_norm(focus_safe)
    if "decoder" not in focus_safe.columns:
        focus_safe["decoder"] = "unknown"

    status_pipe = FilterPipeline(focus_safe, label="status_filter")
    status_pipe = status_pipe.filter(
        status_pipe.result()["run_status_norm"] == "completed",
        label="completed_only", reason="run_status_norm == completed"
    )
    ok_rows = status_pipe.result()
    ok_rows = ensure_numeric_columns(ok_rows, required_numeric_cols)
    if "run_id" not in ok_rows.columns:
        ok_rows["run_id"] = np.nan
    if "decoder" not in ok_rows.columns:
        ok_rows["decoder"] = "unknown"

    if ok_rows.empty:
        panel = pd.DataFrame()
        display(unavailable_panel_table(
            panel="Structure regime comparison",
            reason="No completed rows available after status filtering.",
        ))
    else:
        if ok_rows["structure_collision_rate"].notna().sum() < 3:
            ok_rows["collision_regime"] = "unavailable"
        else:
            ok_rows["collision_regime"] = pd.qcut(
                ok_rows["structure_collision_rate"],
                q=3,
                labels=["low", "mid", "high"],
                duplicates="drop",
            )

        ok_rows["rank_regime"] = np.where(
            ok_rows["structure_rank_deficiency"].fillna(0) > 0,
            "rank_deficient",
            "rank_full",
        )
        ok_rows["structure_regime"] = (
            ok_rows["collision_regime"].astype(str) + "|" + ok_rows["rank_regime"].astype(str)
        )

        panel = (
            ok_rows.groupby(["decoder", "structure_regime"], as_index=False)
            .agg(
                n=("run_id", "size"),
                mean_best_sampled_F=("best_sampled_F", "mean"),
                mean_postselection_success=("postselection_success", "mean"),
                mean_gain_over_bp1=("gain_over_bp1", "mean"),
                mean_gain_over_bruteforce=("gain_over_bruteforce", "mean"),
            )
            .sort_values(["decoder", "structure_regime"])
        )

        if panel.empty:
            display(unavailable_panel_table(
                panel="Structure regime comparison",
                reason="No completed rows available after regime aggregation.",
            ))
        else:
            display(panel)

# Show explicit placeholders for decoder/regime combinations with no rows.
expected_decoders = sorted(focus.get("decoder", pd.Series(dtype=object)).fillna("unknown").astype(str).unique()) if not focus.empty else []
expected_regimes = sorted(panel["structure_regime"].dropna().unique()) if not panel.empty else []
missing_pairs = []
for d in expected_decoders:
    for r in expected_regimes:
        if panel[(panel["decoder"].astype(str) == str(d)) & (panel["structure_regime"] == r)].empty:
            missing_pairs.append({"decoder": d, "structure_regime": r, "status": "unavailable"})
if missing_pairs:
    display(pd.DataFrame(missing_pairs))


## Decoder Gain vs Structure Metrics


In [ ]:
corr_targets = [
    "structure_collision_rate",
    "structure_ambiguous_syndrome_count",
    "structure_short_cycle_proxy",
    "structure_rank_deficiency",
    "structure_distance_proxy",
]
corr_numeric_cols = corr_targets + ["gain_over_bp1", "gain_over_bruteforce"]

if "ok_rows" in globals():
    ok_rows = ensure_numeric_columns(ok_rows, corr_numeric_cols)
else:
    ok_rows = pd.DataFrame(columns=corr_numeric_cols)

if "decoder" not in ok_rows.columns:
    ok_rows["decoder"] = "unknown"

gain_plot_df = ok_rows.copy()
gain_plot_df = unavailable_or_numeric(gain_plot_df, "gain_over_bp1") if not gain_plot_df.empty else gain_plot_df

if gain_plot_df.empty:
    fig, _ax = placeholder_plot("Gain vs Collision Rate", "No completed rows available.")
    display(fig)
    plt.close(fig)
else:
    valid = gain_plot_df[
        gain_plot_df["gain_over_bp1"].notna() & gain_plot_df["structure_collision_rate"].notna()
    ].copy()
    if valid.empty:
        fig, _ax = placeholder_plot("Gain vs Collision Rate", "`gain_over_bp1` or `structure_collision_rate` unavailable.")
        display(fig)
        plt.close(fig)
    else:
        fig, ax = plt.subplots(figsize=(8.5, 4.0))
        for decoder, sub in valid.groupby("decoder", dropna=False):
            label = "unknown" if pd.isna(decoder) else str(decoder)
            ax.scatter(
                sub["structure_collision_rate"],
                sub["gain_over_bp1"],
                label=label,
                alpha=0.85,
                s=36,
            )
        ax.axhline(0.0, color="black", linewidth=1)
        ax.set_title("Decoder gain over BP1 vs collision rate")
        ax.set_xlabel("structure_collision_rate")
        ax.set_ylabel("gain_over_bp1")
        ax.grid(alpha=0.3)
        if ax.collections:
            ax.legend(loc="best")
        fig.tight_layout()
        display(fig)
        plt.close(fig)

corr_rows = []
for metric in corr_targets:
    for gain in ["gain_over_bp1", "gain_over_bruteforce"]:
        x = ok_rows[metric]
        y = ok_rows[gain]
        valid = x.notna() & y.notna()
        if valid.sum() >= 3:
            rho = float(pd.Series(x[valid]).corr(pd.Series(y[valid]), method="spearman"))
            corr_rows.append({"metric": metric, "gain_metric": gain, "n": int(valid.sum()), "spearman_rho": rho, "status": "available"})
        else:
            corr_rows.append({"metric": metric, "gain_metric": gain, "n": int(valid.sum()), "spearman_rho": np.nan, "status": "unavailable"})

corr_df = pd.DataFrame(corr_rows)
display(corr_df)


## Unavailable / Skipped Cases


In [ ]:
slot_df = pd.DataFrame(run_manifest.get("slots", [])) if isinstance(run_manifest, dict) else pd.DataFrame()
requested_slots = slot_df[
    (slot_df.get("matrix", pd.Series(dtype=str)) == "matrix_c")
    & (slot_df.get("stage", pd.Series(dtype=str)) == "decoder_study")
    & (slot_df.get("family", pd.Series(dtype=str)).isin(["P", "S"]))
] if not slot_df.empty else pd.DataFrame()

focus_for_status = ensure_run_status_norm(focus.copy())
for col in ["instance_id", "family", "decoder", "alpha_mode", "run_status", "run_status_norm", "status_reason_code", "run_error"]:
    if col not in focus_for_status.columns:
        focus_for_status[col] = np.nan

status_cols = [
    "instance_id", "family", "decoder", "alpha_mode", "run_status", "run_status_norm", "status_reason_code", "run_error"
]
status_cols = [c for c in status_cols if c in focus_for_status.columns]

not_in_manifest = requested_slots.empty
not_in_matrix_rows = focus_for_status[
    focus_for_status["run_status_norm"] == "not_in_experiment_matrix"
][status_cols].copy() if not focus_for_status.empty else pd.DataFrame(columns=status_cols)
not_applicable_rows = focus_for_status[
    focus_for_status["run_status_norm"].isin(["not_applicable", "unavailable"])
][status_cols].copy() if not focus_for_status.empty else pd.DataFrame(columns=status_cols)
failed_rows = focus_for_status[
    focus_for_status["run_status_norm"] == "failed"
][status_cols].copy() if not focus_for_status.empty else pd.DataFrame(columns=status_cols)

summary_rows = [{
    "bucket": "not_in_experiment_matrix",
    "count": int(len(not_in_matrix_rows)) if len(not_in_matrix_rows) > 0 else int(not_in_manifest),
    "details": (
        "Rows marked not_in_experiment_matrix in metrics_master."
        if len(not_in_matrix_rows) > 0 else
        ("No decoder_study slots in manifest for matrix_c P/S." if not_in_manifest else "present")
    ),
}, {
    "bucket": "not_applicable_or_unavailable",
    "count": int(len(not_applicable_rows)),
    "details": "Rows normalized to not_applicable/unavailable.",
}, {
    "bucket": "failed",
    "count": int(len(failed_rows)),
    "details": "Rows attempted but errored.",
}]
display(pd.DataFrame(summary_rows))

if not not_in_matrix_rows.empty:
    display(not_in_matrix_rows)
if not not_applicable_rows.empty:
    display(not_applicable_rows)
if not failed_rows.empty:
    display(failed_rows)
if not_in_matrix_rows.empty and not_applicable_rows.empty and failed_rows.empty and not not_in_manifest:
    display(unavailable_panel_table(
        panel="Unavailable / skipped cases",
        reason="No non-completed status rows detected in this slice.",
    ))

unavailable_metrics = []
for _, row in focus_for_status.iterrows():
    avail = row.get("metric_availability_dict", {})
    if not isinstance(avail, dict):
        continue
    for metric, status in avail.items():
        if status != "available":
            unavailable_metrics.append({
                "instance_id": row.get("instance_id"),
                "decoder": row.get("decoder"),
                "alpha_mode": row.get("alpha_mode"),
                "metric": metric,
                "status": status,
            })
if unavailable_metrics:
    display(pd.DataFrame(unavailable_metrics))


## Conclusion (Gap 2)


In [ ]:
if 'corr_df' not in globals() or corr_df.empty:
    display(Markdown("- Gap 2 evidence is limited because decoder-study correlation rows are unavailable."))
else:
    available_corr = corr_df[corr_df["status"] == "available"].copy()
    if available_corr.empty:
        display(Markdown("- Gap 2 evidence is partial: stage `decoder_study` rows are present, but gain/structure correlations are unavailable for this slice."))
    else:
        # Check if there are any non-NaN spearman_rho values
        valid_rho = available_corr["spearman_rho"].notna()
        if not valid_rho.any():
            display(Markdown("- Gap 2 evidence is partial: correlation rows are available but all spearman_rho values are unavailable."))
        else:
            strongest = available_corr.loc[valid_rho].iloc[available_corr.loc[valid_rho, "spearman_rho"].abs().idxmax()]
            display(Markdown(
                f"- Stage C decoder-study rows show non-uniform decoder behavior across structure regimes; strongest observed correlation is `rho={strongest['spearman_rho']:.3f}` for `{strongest['gain_metric']}` vs `{strongest['metric']}`.\n"
                "- Notebook placeholders explicitly distinguish `not_in_experiment_matrix`, `not_applicable`, and `failed` slices."
            ))
